# Preprocess

In [1]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import pandas as pd
import numpy as np
import muon

2024-12-16 02:22:56.492235: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-16 02:22:56.531391: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-16 02:22:56.531448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-16 02:22:56.533185: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-16 02:22:56.540785: I tensorflow/core/platform/cpu_feature_guar

In [2]:
import os 
os.chdir('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/')

In [3]:
rna = sc.read('./processed/rna_hvg/rna_hvg.h5ad')
atac = sc.read('./processed/atac_hvg/atac_hvf.h5ad')

In [6]:
rna.X

<1641866x4000 sparse matrix of type '<class 'numpy.float32'>'
	with 531251953 stored elements in Compressed Sparse Row format>

In [5]:
atac.X

<1641866x60000 sparse matrix of type '<class 'numpy.float64'>'
	with 161108382 stored elements in Compressed Sparse Row format>

## RNA-seq preprocessing

In [4]:
rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
rna

AnnData object with n_obs × n_vars = 1641866 × 36601
    obs: 'dataset', 'barcode', 'sample'

In [5]:
rna.layers['counts'] = rna.X.copy()

In [6]:
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

In [7]:
sc.pp.highly_variable_genes(rna, n_top_genes=4000, batch_key='sample')
rna

AnnData object with n_obs × n_vars = 1641866 × 36601
    obs: 'dataset', 'barcode', 'sample'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'log1p', 'hvg'
    layers: 'counts'

In [8]:
rna_hvg = rna[:, rna.var.highly_variable].copy()
rna_hvg

AnnData object with n_obs × n_vars = 1641866 × 4000
    obs: 'dataset', 'barcode', 'sample'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'log1p', 'hvg'
    layers: 'counts'

In [9]:
rna.write('./processed/rna_hvg/rna.h5ad', compression='gzip')
rna_hvg.write('./processed/rna_hvg/rna_hvg.h5ad', compression='gzip')

## ATAC-seq preprocessing

In [3]:
atac = sc.read_h5ad('./processed/atac/data_new.h5ad')
atac

AnnData object with n_obs × n_vars = 1641866 × 527459
    obs: 'dataset', 'barcode', 'sample'

In [4]:
atac.layers['counts'] = atac.X.copy()

In [ ]:
muon.atac.pp.tfidf(atac, scale_factor=1e4)

In [ ]:
sc.pp.highly_variable_genes(atac, n_top_genes=60000, batch_key='sample')
atac

In [ ]:
atac_hvf = atac[:, atac.var.highly_variable].copy()
atac_hvf

In [ ]:
atac.write('./processed/atac_hvg/atac.h5ad', compression='gzip')
atac_hvf.write('./processed/atac_hvg/atac_hvf.h5ad', compression='gzip')

## MultiVI

In [4]:
rna = sc.read_h5ad('./processed/rna_hvg/rna_hvg.h5ad')
atac = sc.read_h5ad('./processed/atac_hvg/atac_hvf.h5ad')

In [6]:
import anndata as ad

In [7]:
merged_adata = ad.concat([rna, atac], axis=1) 
merged_adata

AnnData object with n_obs × n_vars = 1641866 × 64000
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    layers: 'counts'

In [8]:
merged_adata.obs = rna.obs
merged_adata

AnnData object with n_obs × n_vars = 1641866 × 64000
    obs: 'dataset', 'barcode', 'sample'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    layers: 'counts'

In [9]:
merged_adata.var['modality'] = ['0'] * 4000 + ['1'] * 60000

In [10]:
merged_adata.write('./processed/rna_atac_hvg.h5ad', compression='gzip')

## UINMF

In [3]:
data = sc.read_h5ad('./processed/rna_atac_hvg.h5ad')
data

AnnData object with n_obs × n_vars = 1641866 × 64000
    obs: 'dataset', 'barcode', 'sample'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'modality'
    layers: 'counts'

In [9]:
from scipy.io import mmwrite
from scipy.sparse import csr_matrix

mmwrite("./processed/uinmf_matrix_norm.mtx", data.X)

In [5]:
cell = pd.DataFrame(data.obs_names)
cell

,0
0,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1
1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1
2,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1
3,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1
4,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1
...,...
1641861,Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C
1641862,Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C
1641863,Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C
1641864,Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C


In [6]:
cell.to_csv('./processed/uinmf_cell.csv')

In [7]:
gene = pd.DataFrame(data.var_names)
gene

,0
0,C1orf159
1,TNFRSF18
2,TNFRSF4
3,PRDM16
4,AJAP1
...,...
63995,chrX-155231109-155231609
63996,chrX-155534736-155535236
63997,chrX-155869724-155870224
63998,chrX-155873118-155873618


In [8]:
gene.to_csv('./processed/uinmf_feature.csv')

## wnn-rna

In [6]:
# embed = pd.DataFrame(np.load('./processed/integrate/rna/rna_pca.npy'))
embed = pd.DataFrame(np.load('./processed/integrate/rna/rna_pca_harmony.npy'))
embed

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,-2.966014,-0.937564,-0.896205,-1.299390,0.882937,0.233117,3.803258,0.145394,-2.376795,-2.398846,...,-1.235092,0.285206,-0.069341,0.636265,-0.940759,0.876321,0.736136,0.683070,-1.225456,0.055319
1,-2.955009,-2.323029,-1.557704,0.061971,0.119083,-0.075018,1.928747,-0.888774,-3.970664,-4.092036,...,-0.786007,1.808690,0.650357,1.110872,-0.698767,-0.773390,-0.963591,0.712412,-0.565072,-0.217342
2,-6.420721,0.119557,-1.377429,-0.461959,-0.253910,-0.021700,2.447033,0.450332,-0.193178,-2.395282,...,0.165633,-0.496469,0.871389,-0.397360,1.820016,0.888384,-3.319887,-0.852245,1.268730,-0.738463
3,-4.292026,0.939285,-2.783409,0.317620,4.052634,-0.541714,1.196937,0.396208,1.266291,-0.734940,...,0.312795,0.820882,1.587937,-0.716955,0.535824,-1.624801,0.162743,-0.283248,-0.440609,-1.220211
4,-4.168028,-2.120243,-1.469557,-1.205668,-1.639010,-0.916777,-1.149948,-1.203078,-1.926339,-1.564186,...,3.243036,-0.502255,-0.523723,-0.871304,-0.124733,0.071028,0.914296,1.513425,1.675982,0.571817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1641861,-3.307714,1.479048,-4.377657,0.168519,11.319915,-3.962230,-1.095084,3.202113,2.692813,2.705274,...,-0.234132,0.548963,-0.288717,-1.731171,1.439535,-0.075492,-1.276867,0.816642,-0.051675,-1.368437
1641862,12.315427,11.972909,-4.188318,-1.046476,-2.448480,2.804988,0.014719,1.656585,0.074217,2.242438,...,0.087135,-0.942688,0.653842,1.149305,-1.741871,0.002188,1.922501,-0.274481,-0.048617,0.532137
1641863,7.516565,-2.797602,1.175496,5.816702,1.044103,0.379200,1.260210,-1.003564,-1.798470,1.860750,...,-0.086160,1.211563,0.707463,0.575511,-0.313417,0.192307,0.406126,-0.136139,-0.196975,-0.646316
1641864,10.499887,-8.266982,1.799513,14.192731,-0.858032,-0.581402,2.334313,2.182132,5.763277,0.179457,...,2.307614,2.276566,0.133975,0.694645,-0.549695,0.680594,1.659606,-0.569295,0.180954,-1.422580


In [7]:
# embed.to_csv('./processed/integrate/wnn/rna/rna_pca.csv')
embed.to_csv('./processed/integrate/wnn/rna/rna_pca_harmony.csv')

## wnn-atac

In [10]:
# embed = pd.DataFrame(np.load('./processed/integrate/atac/atac_lsi.npy'))
embed = pd.DataFrame(np.load('./processed/integrate/atac/atac_lsi_harmony.npy'))
embed

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,-0.168629,-0.083156,0.279250,-0.193976,-0.237838,0.035171,-0.350788,0.037151,0.084135,-0.545958,...,-0.659046,-0.547487,0.028206,-0.015292,-0.648967,0.715672,-0.141283,0.069430,0.146149,-0.420941
1,-0.043728,0.319210,0.153306,-0.075416,-0.131133,0.152023,-0.065417,0.151521,0.172131,-0.563011,...,-0.469238,0.792593,-0.173229,0.633509,-0.619624,0.587517,0.265741,0.390472,-0.988708,0.253413
2,-0.337461,-0.025174,0.151155,-0.185322,-0.029148,-0.267411,-0.204810,-0.292993,-0.275500,-0.235437,...,0.153173,0.228062,0.314035,-0.331684,-0.395489,-0.433686,0.068826,0.317408,-0.910065,0.125350
3,-0.151810,-0.058167,0.216861,0.014258,-0.284697,-0.052832,0.059763,-0.356265,-0.121769,-0.435497,...,-0.490408,0.670274,0.282336,0.792767,-0.584786,0.513373,0.513076,0.200436,-0.951790,0.264028
4,-0.110210,-0.121232,0.273665,-0.320714,-0.377898,0.135706,-0.285467,-0.136178,-0.199255,-0.568376,...,0.025228,0.260428,-0.056620,-0.616125,0.064770,0.336239,0.262472,0.240132,-0.224383,-0.121606
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1641861,0.079926,0.193893,-0.192612,0.938236,1.376493,-0.357191,0.172663,1.641718,0.616751,-0.178761,...,-0.545822,0.077016,-0.274030,-1.128947,-0.029912,-0.051544,0.210034,0.670567,-0.009610,-0.268650
1641862,0.549913,0.277919,-0.453313,0.219317,-0.151764,0.635382,-0.012173,0.165442,-0.007258,-0.132963,...,-0.137254,-0.065540,-0.239453,0.579347,0.421047,-0.214972,-1.044484,0.537430,-0.065976,-0.022710
1641863,-1.854301,-0.618379,-0.982233,-1.112775,-0.680338,-1.567807,1.201358,0.742316,-1.792531,0.819453,...,-0.275760,0.600273,-0.350309,0.934719,0.830009,0.448049,1.085043,-1.608526,0.293582,0.235492
1641864,-0.579517,-0.033013,0.380179,-0.238633,0.115657,-0.844177,1.490168,0.041096,0.157108,0.505995,...,-1.207846,0.851968,-0.460980,-0.470313,-0.179657,1.021049,-0.969605,0.176996,-0.135132,1.343115


In [11]:
# embed.to_csv('./processed/integrate/wnn/atac/atac_lsi.csv')
embed.to_csv('./processed/integrate/wnn/atac/atac_lsi_harmony.csv')

# compute

## integrate_atac

In [ ]:
import snapatac2 as snap
import scanpy as sc
import numpy as np
import muon
import scanpy.external as sce

# spectral
rna = sc.read_h5ad('../processed/atac/data_new.h5ad')
# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

# np.save('atac_spectral.npy', rna.obsm['X_spectral'])
# np.save('atac_spectral_umap.npy', rna.obsm['X_umap'])

rna.obsm['X_pca'] = np.load('../processed/integrate/atac/atac_spectral.npy')
sce.pp.harmony_integrate(rna, 'sample')
rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

np.save('atac_spectral_harmony.npy', rna.obsm['X_pca'])
# np.save('atac_spectral_harmony_umap.npy', rna.obsm['X_umap'])


# tfidf
# rna = sc.read_h5ad('./processed/atac/data_new.h5ad')
# muon.atac.tl.lsi(rna)
# rna.obsm['X_pca'] = rna.obsm['X_lsi'][:,1:]
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=40)
# sc.tl.umap(rna)

# np.save('atac_lsi.npy', rna.obsm['X_pca'])
# np.save('atac_lsi_umap.npy', rna.obsm['X_umap'])

# rna.obsm['X_pca'] = np.load('atac_lsi.npy')
# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=40)
# sc.tl.umap(rna)

# np.save('atac_lsi_harmony.npy', rna.obsm['X_pca'])
# np.save('atac_lsi_harmony_umap.npy', rna.obsm['X_umap'])

## integrate_rna

In [ ]:
import snapatac2 as snap
import scanpy as sc
import numpy as np
import scanpy.external as sce

# # spectral1
# rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
# sc.pp.highly_variable_genes(rna, flavor='seurat_v3', n_top_genes=3000)
# rna = rna[:, rna.var.highly_variable]
# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# snap.tl.spectral(rna, features=None)
# rna.obsm['X_pca'] = rna.obsm['X_spectral']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('rna_spectral1.npy', rna.obsm['X_pca'])
# np.save('rna_spectral1_umap.npy', rna.obsm['X_umap'])

# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('rna_spectral1_harmony.npy', rna.obsm['X_pca'])
# np.save('rna_spectral1_harmony_umap.npy', rna.obsm['X_umap'])

# # spectral2
# rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
# rna = rna[:, rna.var.highly_variable]
# snap.tl.spectral(rna, features=None)
# rna.obsm['X_pca'] = rna.obsm['X_spectral']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('rna_spectral2.npy', rna.obsm['X_pca'])
# np.save('rna_spectral2_umap.npy', rna.obsm['X_umap'])

# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('rna_spectral2_harmony.npy', rna.obsm['X_pca'])
# np.save('rna_spectral2_harmony_umap.npy', rna.obsm['X_umap'])

# pca
rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
rna = rna[:, rna.var.highly_variable]
sc.pp.scale(rna, max_value=10)
sc.tl.pca(rna, svd_solver='arpack')
sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
sc.tl.umap(rna)

np.save('rna_pca.npy', rna.obsm['X_pca'])
np.save('rna_pca_umap.npy', rna.obsm['X_umap'])

sce.pp.harmony_integrate(rna, 'sample')
rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
sc.tl.umap(rna)

np.save('rna_pca_harmony.npy', rna.obsm['X_pca'])
np.save('rna_pca_harmony_umap.npy', rna.obsm['X_umap'])

## multi_spectral

In [ ]:
import snapatac2 as snap
import scanpy as sc
import numpy as np
import scanpy.external as sce

# atac = sc.read_h5ad('./processed/atac/data_new.h5ad')
# snap.tl.spectral(atac, features=None)
# snap.tl.umap(atac)

# rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
# rna = rna[:, rna.var.highly_variable]
# snap.tl.spectral(rna, features=None)

# assert (rna.obs_names == atac.obs_names).all()
# embedding = snap.tl.multi_spectral([rna, atac], features=None)[1]
# np.save('multi_spectral.npy', embedding)
# atac.obsm['X_joint'] = embedding
# snap.tl.umap(atac, use_rep='X_joint')
# np.save('multi_spectral_umap.npy', atac.obsm['X_umap'])

# embedding = np.load('./processed/integrate/multi_spectral/multi_spectral.npy')
# rna.obsm['X_pca'] = embedding
# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('multi_spectral_harmony.npy', rna.obsm['X_pca'])
# np.save('multi_spectral_harmony_umap.npy', rna.obsm['X_umap'])

# harmony
atac = sc.read_h5ad('./processed/atac/data_new.h5ad')
# snap.tl.spectral(atac, features=None)
# snap.tl.umap(atac)
# atac.obsm['X_pca'] = atac.obsm['X_spectral']
# sce.pp.harmony_integrate(atac, 'sample')
# atac.obsm['X_spectral'] = atac.obsm['X_pca_harmony']
embed = np.load('./processed/integrate/atac/atac_spectral_harmony.npy')
atac.obsm['X_spectral'] = embed

rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
# rna = rna[:, rna.var.highly_variable]
# snap.tl.spectral(rna, features=None)
# rna.obsm['X_pca'] = rna.obsm['X_spectral']
# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_spectral'] = rna.obsm['X_pca_harmony']
embed = np.load('./processed/integrate/rna/rna_spectral2_harmony.npy')
rna.obsm['X_spectral'] = embed

assert (rna.obs_names == atac.obs_names).all()
embedding = snap.tl.multi_spectral([rna, atac], features=None)[1]
np.save('harmony_multi_spectral.npy', embedding)
atac.obsm['X_joint'] = embedding
snap.tl.umap(atac, use_rep='X_joint')
np.save('harmony_multi_spectral_umap.npy', atac.obsm['X_umap'])

## RunMOFA

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from scipy.io import mmread
from scipy.sparse import csr_matrix
import os, sys, time
from muon import MuData
import importlib
# sys.path.append(".")
# sys.path.append("./mofapy2/")
import mofapy2

rna = sc.read('./processed/rna_hvg/rna_hvg.h5ad')
sc.pp.subsample(rna, 0.2)
atac = sc.read('./processed/atac_hvg/atac_hvf.h5ad')
atac = atac[rna.obs_names.tolist()]

mdata = MuData({'rna': rna, 'atac': atac})
mdata.obs['batch'] = rna.obs['sample'].tolist()
del rna, atac

mu.tl.mofa(mdata,use_var=None, groups_label="batch", gpu_mode=True,save_data=False)
sc.pp.neighbors(mdata, use_rep="X_mofa")
sc.tl.umap(mdata)

mdata.write('mofa.h5mu')
np.save("mofa.np", mdata.obsm['X_mofa'])
np.save("mofa_umap.np", mdata.obsm['X_umap'])

# Limited by memory, 60k cells require 141G
# Benchmarking algorithms for single-cell multi-omics prediction and integration
# subsample, 0.2

## RunMultigrate

In [ ]:
from scipy.io import mmread
from scipy.sparse import csr_matrix
# import scarches as sca
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import muon
# import gdown
import os, sys, time
# import importlib
# importlib.reload(sca)

import torch
import warnings
warnings.filterwarnings("ignore")

rna = sc.read('./processed/rna_hvg/rna_hvg.h5ad')
atac = sc.read('./processed/atac_hvg/atac_hvf.h5ad')
rna.layers['norm'] = rna.X
atac.layers['tfidf'] = atac.X


import multigrate as mtg

adata = mtg.data.organize_multiome_anndatas(
    adatas = [[rna], [atac]],    # a list of anndata objects per modality, RNA-seq always goes first
    layers = [['norm'], ['tfidf']]
)
del rna, atac

mtg.model.MultiVAE.setup_anndata(
    adata,
    categorical_covariate_keys=['sample'],
    rna_indices_end=4000,
)

model = mtg.model.MultiVAE(
    adata,
    losses=['mse', 'mse']
)

model.train(use_gpu=True)
torch.save(model, 'mtg_model.pth')

adata.obsm['X_pca'] = model.get_latent_representation()
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=30)
sc.tl.umap(adata)

np.save('mtg.npy', adata.obsm['X_pca'])
np.save('mtg_umap.npy', adata.obsm['X_umap'])

## RunMultiVI

In [ ]:
from scipy.io import mmread
from scipy.sparse import csr_matrix
import anndata as ad
import pandas as pd
import scvi
import numpy as np
import scanpy as sc
scvi.settings.seed = 420
import os, sys, time
import torch

adata_mvi = sc.read_h5ad('./processed/integrate/rna_atac_hvg.h5ad')
# adata_mvi.X = adata_mvi.layers['counts'].copy()

# scvi.model.MULTIVI.setup_anndata(adata_mvi, batch_key="sample")

# model = scvi.model.MULTIVI(
#     adata_mvi,
#     n_genes=(adata_mvi.var["modality"] == "0").sum(),
#     n_regions=(adata_mvi.var["modality"] == "1").sum(),
# )
# model.view_anndata_setup()

# model.train(use_gpu=True)
# torch.save(model, 'mtv_model.pth')
model = torch.load('mtv_model.pth')
adata_mvi.obsm['X_pca'] = model.get_latent_representation()
sc.pp.neighbors(adata_mvi, n_neighbors=10, n_pcs=15)
sc.tl.umap(adata_mvi)

np.save('mtv.npy', adata_mvi.obsm['X_pca'])
np.save('mtv_umap.npy', adata_mvi.obsm['X_umap'])

## UINMF

In [ ]:
# uinmf_create_object
library(Matrix)
library(rliger)
setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/')

matrix <- readMM('processed/uinmf/uinmf_matrix_norm.mtx')
matrix <- as(matrix, 'dgCMatrix')
matrix <- t(matrix)

cell <- read.csv('processed/uinmf/uinmf_cell.csv', row.names = 1)
feature <- read.csv('processed/uinmf/uinmf_feature.csv', row.names = 1)

colnames(matrix) <- cell$X0
rownames(matrix) <- feature$X0

df <- as.data.frame(do.call(rbind, strsplit(unlist(cell$X0), "-")), stringsAsFactors = FALSE)
df$V4 <- paste0(df$V1, '-', df$V3)

rownames(df) <- cell$X0
sample <- as.data.frame(table(df$V4))

lis <- list()
for (i in 1:201){
  mat <- matrix[,rownames(df[df$V4 == sample[i,1],])]
  lis$sam <- mat
  names(lis)[names(lis) == "sam"] <- as.character(sample[i,1])
}

liger <- createLiger(lis, 
                     remove.missing = FALSE,
                     addPrefix = FALSE)
for (i in names(liger@datasets)) {
  liger@datasets[[i]]@normData <- liger@datasets[[i]]@rawData
}

liger@varFeatures <- feature$X0
liger <- scaleNotCenter(liger)

saveRDS(liger, 'processed/uinmf/liger.rds')

In [ ]:
# uinmf_integration
library(Matrix)
library(rliger)
setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/')

liger <- readRDS('processed/uinmf/liger.rds')

liger <- runIntegration(liger, k = 30)

liger <- quantileNorm(liger)
liger <- runCluster(liger)
liger <- runUMAP(liger)

saveRDS(liger, 'processed/uinmf/liger_integration.rds')

# k20, online

# liger_integration <- readRDS("/pool1/qiaolibo/R_Project/peak_gene/z_others/7_horizontal_RNA_ATAC/processed/uinmf/liger_integration.rds")
# 
# norm_H = liger_integration@H.norm
# rownames(norm_H) <- colnames(liger)
# write.csv(norm_H, 'results/blood/uinmf_latent.csv')
# 
# umap = liger_integration@dimReds[["UMAP"]]
# rownames(umap) <- colnames(liger)
# write.csv(umap, 'results/blood/uinmf_umap.csv')

# k30
#liger_integration_k30 <- readRDS("processed/uinmf/liger_integration_k30.rds")

norm_H = liger@H.norm
write.csv(norm_H, 'processed/uinmf/uinmf.csv')

umap = liger@dimReds[["UMAP"]]
write.csv(umap, 'processed/uinmf/uinmf_umap.csv')

## wnn

In [ ]:
# wnn_create_object
library(Matrix)
library(Seurat)
setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/integrate/wnn/')

sparse_mat <- sparseMatrix(i = integer(0), j = integer(0), dims = c(2,1641866))
colnames(sparse_mat) <- 1:1641866
rownames(sparse_mat) <- 1:2
pbmc <- CreateSeuratObject(counts = sparse_mat)

pbmc@reductions[["pca"]] <- wnn@reductions[["pca"]]
pbmc@reductions[["lsi"]] <- wnn@reductions[["pca"]]

# rna <- read.csv('rna/rna_pca.csv', header = T, row.names = 1)
# rownames(rna) <- 1:1641866
# pbmc@reductions[["pca"]]@cell.embeddings <- as.matrix(rna)
# rna <- read.csv('atac/atac_lsi.csv', header = T, row.names = 1)
# rownames(rna) <- 1:1641866
# pbmc@reductions[["lsi"]]@cell.embeddings <- as.matrix(rna)

rna <- read.csv('rna/rna_pca_harmony.csv', header = T, row.names = 1)
rownames(rna) <- 1:1641866
pbmc@reductions[["pca"]]@cell.embeddings <- as.matrix(rna)
rna <- read.csv('atac/atac_lsi_harmony.csv', header = T, row.names = 1)
rownames(rna) <- 1:1641866
pbmc@reductions[["lsi"]]@cell.embeddings <- as.matrix(rna)

saveRDS(pbmc,'wnn_h.rds')

In [ ]:
# wnn_h_m
library(Matrix)
library(Seurat)
library(future)

options(future.globals.maxSize = 2 * 1024^3)  # 设置为 2GB

future::plan(future::multisession, workers = 4)

setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/integrate/wnn/')

pbmc <- readRDS('wnn_h.rds')
colnames(pbmc@reductions[["pca"]]@cell.embeddings) <- paste0("PC_", 1:50)
colnames(pbmc@reductions[["lsi"]]@cell.embeddings) <- paste0("PC_", 1:49)
pbmc <- FindMultiModalNeighbors(pbmc, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 1:49))
pbmc <- RunUMAP(pbmc, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")

saveRDS(pbmc,'wnn_h_m.rds')

In [ ]:
# wnn_m
library(Matrix)
library(Seurat)
library(future)

options(future.globals.maxSize = 2 * 1024^3)  # 设置为 2GB

future::plan(future::multisession, workers = 4)

setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/integrate/wnn/')

pbmc <- readRDS('wnn.rds')
colnames(pbmc@reductions[["pca"]]@cell.embeddings) <- paste0("PC_", 1:50)
colnames(pbmc@reductions[["lsi"]]@cell.embeddings) <- paste0("PC_", 1:49)
pbmc <- FindMultiModalNeighbors(pbmc, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 1:49))
pbmc <- RunUMAP(pbmc, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")

saveRDS(pbmc,'wnn_m.rds')

# visual

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
sc.settings.set_figure_params(dpi=100, dpi_save=300, facecolor="white")
plt.rcParams['font.sans-serif'] = ['DejaVu Sans'] 

In [3]:
data = sc.read_h5ad('./processed/adata.h5ad')
data

AnnData object with n_obs × n_vars = 1641866 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType_raw', 'cellType_raw_level1', 'cellType_raw_level2', 'cellType_raw_level3', 'cellType_scarch_level3', 'cellType_actinn_level3', 'cellType_celltypist_level3', 'cellType_scmap_level3', 'cellType_singlecellnet_level3', 'cellType_scpred_level3', 'cellType_singler_level3', 'cellType_chetah_level3', 'cellType_scarch_level2', 'cellType_actinn_level2', 'cellType_celltypist_level2', 'cellType_scmap_level2', 'cellType_singlecellnet_level2', 'cellType_scpred_level2', 'cellType_singler_level2', 'cellType_chetah_level2', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'final_ann'

## rna

In [ ]:
umap = np.load('./processed/integrate/rna/rna_spectral2_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_rna_spectral2_umap')

In [ ]:
umap = np.load('./processed/integrate/rna/rna_spectral2_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_rna_spectral2_harmony_umap')

In [ ]:
umap = np.load('./processed/integrate/rna/rna_pca_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_rna_pca_umap')

In [ ]:
umap = np.load('./processed/integrate/rna/rna_pca_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_rna_pca_harmony_umap')

## atac

In [ ]:
umap = np.load('./processed/integrate/atac/atac_spectral_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_atac_spectral_umap')

In [ ]:
umap = np.load('./processed/integrate/atac/atac_spectral_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_atac_spectral_harmony_umap')

In [ ]:
umap = np.load('./processed/integrate/atac/atac_lsi_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_atac_lsi_umap')

In [ ]:
umap = np.load('./processed/integrate/atac/atac_lsi_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_atac_lsi_harmony_umap')

## multi_spectral

In [ ]:
umap = np.load('./processed/integrate/multi_spectral/multi_spectral_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_multi_spectral_umap')

In [ ]:
umap = np.load('./processed/integrate/multi_spectral/multi_spectral_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_multi_spectral_harmony_umap')

In [ ]:
umap = np.load('./processed/integrate/multi_spectral/harmony_multi_spectral_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_harmony_multi_spectral_umap')

## wnn

In [ ]:
umap = pd.read_csv('./processed/integrate/wnn/wnn_m_umap.csv', index_col=0).to_numpy()
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_wnn_m_umap')

In [ ]:
umap = pd.read_csv('./processed/integrate/wnn/wnn_h_m_umap.csv', index_col=0).to_numpy()
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_wnn_h_m_umap')

## joint

In [ ]:
# multivi
umap = np.load('./processed/integrate/multivi/mtv_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_mtv')

In [6]:
data.uns

OrderedDict([('final_ann_colors',
              ['#1f77b4',
               '#ff7f0e',
               '#279e68',
               '#d62728',
               '#aa40fc',
               '#8c564b',
               '#e377c2',
               '#b5bd61',
               '#17becf',
               '#aec7e8',
               '#ffbb78',
               '#98df8a',
               '#ff9896',
               '#c5b0d5'])])

In [25]:
# uinmf
umap = pd.read_csv('./processed/integrate/uinmf/uinmf_umap.csv', index_col=0)
umap = umap.reindex(data.obs_names.tolist())
umap

,UMAP_1,UMAP_2
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,-1.313048,-0.691005
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,0.254405,-1.622954
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,-12.224995,3.703995
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,-11.778780,3.841125
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,1.921465,6.470619
...,...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,-4.760157,-6.910507
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,21.224647,3.748355
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,10.801809,0.055916
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,9.691457,0.274131


In [ ]:
data.obsm['X_umap'] = umap.to_numpy()
sc.pl.umap(data, color='final_ann', save='_uinmf_umap')

In [27]:
# mofa
import muon as mu
import mudata as md

/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
mdata = md.read("./processed/integrate/mofa/mofa.h5mu")
mdata

/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 328373 × 64000
  obs:	'batch'
  var:	'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
  uns:	'mofa', 'neighbors', 'umap'
  obsm:	'X_mofa', 'X_umap'
  varm:	'LFs'
  obsp:	'connectivities', 'distances'
  2 modalities
    rna:	328373 x 4000
      obs:	'dataset', 'barcode', 'sample'
      var:	'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'hvg', 'log1p'
      layers:	'counts'
    atac:	328373 x 60000
      obs:	'dataset', 'barcode', 'sample'
      var:	'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'hvg'
      layers:	'counts'

In [29]:
dat = data[mdata.obs_names.to_list()]
dat

View of AnnData object with n_obs × n_vars = 328373 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType_raw', 'cellType_raw_level1', 'cellType_raw_level2', 'cellType_raw_level3', 'cellType_scarch_level3', 'cellType_actinn_level3', 'cellType_celltypist_level3', 'cellType_scmap_level3', 'cellType_singlecellnet_level3', 'cellType_scpred_level3', 'cellType_singler_level3', 'cellType_chetah_level3', 'cellType_scarch_level2', 'cellType_actinn_level2', 'cellType_celltypist_level2', 'cellType_scmap_level2', 'cellType_singlecellnet_level2', 'cellType_scpred_level2', 'cellType_singler_level2', 'cellType_chetah_level2', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'final_ann'
    uns: 'final_ann_colors'
    obsm: 'X_umap'

In [ ]:
umap = np.load('./processed/integrate/mofa/mofa_umap.np.npy')
dat.obsm['X_umap'] = umap
sc.pl.umap(dat, color='final_ann', save='_mofa_umap')

## cell_type

### origin_ann

In [73]:
data.obs['cellType_raw_level2'].value_counts()

cellType_raw_level2
T cells                       786921
notAvailable                  349597
HSC/MPP                       122500
Monocytes                      96756
ILC                            89889
B cells                        75188
Erythroid                      46373
Double-negative thymocytes     21614
Megakaryocytes/platelets       10082
DC                              8437
B-cell lineage                  7959
pDC                             7048
Plasma cells                    4250
Megakaryocyte precursor         3918
Cycling cells                   3427
Macrophages                     2702
Granulocytes                    2684
Mast cells                      1694
DC precursor                     827
Name: count, dtype: int64

In [74]:
data.obs['cellType_raw_level3'].value_counts()

cellType_raw_level3
notAvailable                                    396328
Tem/Effector helper T cells                     294921
Tcm/Naive helper T cells                        225463
Tcm/Naive cytotoxic T cells                      85739
NK cells                                         84177
Classical monocytes                              71776
HSC/MPP                                          67848
Tem/Trm cytotoxic T cells                        52396
B cells                                          48078
Type 1 helper T cells                            45506
Early erythroid                                  28398
Regulatory T cells                               22329
Double-negative thymocytes                       21614
Type 17 helper T cells                           19895
Naive B cells                                    18105
Non-classical monocytes                          17992
Megakaryocyte-erythroid-mast cell progenitor     17134
ELP                                          

In [ ]:
sc.pl.umap(data, color='cellType_raw_level2')

In [96]:
len(data.uns['cellType_raw_level2_colors'])

19

In [97]:
data.uns['cellType_raw_level2_colors'][17] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='cellType_raw_level2')

In [ ]:
sc.pl.umap(data, color='cellType_raw_level3')

In [100]:
len(data.uns['cellType_raw_level3_colors'])

51

In [101]:
data.uns['cellType_raw_level3_colors'][49] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='cellType_raw_level3')

### 6_4

In [89]:
mv_level2 = pd.read_csv('./6_4_level2.csv', index_col=0)
mv_level2

,0
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,notAvailable
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,notAvailable
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,T cells
...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,notAvailable


In [90]:
mv_level3 = pd.read_csv('./6_4_level3.csv', index_col=0)
mv_level3

,0
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,notAvailable
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,notAvailable
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,notAvailable
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,notAvailable
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,notAvailable
...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,notAvailable


In [91]:
data.obs['mv_level2'] = mv_level2['0'].tolist()

In [92]:
data.obs['mv_level3'] = mv_level3['0'].tolist()

In [93]:
data.obs['mv_level2'].value_counts()

mv_level2
T cells                     768427
notAvailable                483933
Monocytes                   148610
B cells                      93330
ILC                          81473
HSC/MPP                      34685
Erythroid                    13385
pDC                           6935
DC                            4634
Plasma cells                  4544
Mast cells                     925
Early MK                       814
Cycling cells                   89
Macrophages                     80
Myelocytes                       1
Megakaryocytes/platelets         1
Name: count, dtype: int64

In [94]:
data.obs['mv_level3'].value_counts()

mv_level3
notAvailable                        1204625
Tcm/Naive helper T cells             162261
Classical monocytes                   58566
CD16+ NK cells                        51747
Naive B cells                         48033
HSC/MPP                               21130
Memory B cells                        17658
Non-classical monocytes               16022
Regulatory T cells                    12450
Early erythroid                        8037
pDC                                    6935
Tcm/Naive cytotoxic T cells            6597
Cytotoxic T cells                      6431
Plasma cells                           4522
GMP                                    3488
MAIT cells                             3142
Helper T cells                         2399
Monocytes                              1788
NK cells                               1567
CD16- NK cells                         1167
Mast cells                              925
Early MK                                814
DC2                   

In [ ]:
sc.pl.umap(data, color='mv_level2')

In [55]:
data.uns['mv_level2_colors'][14] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='mv_level2')

In [ ]:
sc.pl.umap(data, color='mv_level3')

In [58]:
data.uns['mv_level3_colors'][40] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='mv_level3')

### 3_2

In [10]:
mv_level2 = pd.read_csv('./3_2_level2.csv', index_col=0)
mv_level2

,0
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,T cells
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,T cells
...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,ILC
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,T cells
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,notAvailable


In [11]:
mv_level3 = pd.read_csv('./3_2_level3.csv', index_col=0)
mv_level3

,0
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Regulatory T cells
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,notAvailable
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Helper T cells
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Tcm/Naive helper T cells
...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,notAvailable
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,notAvailable


In [12]:
data.obs['mv_level2'] = mv_level2['0'].tolist()

In [13]:
data.obs['mv_level3'] = mv_level3['0'].tolist()

In [ ]:
sc.pl.umap(data, color='mv_level2')

In [15]:
len(data.uns['mv_level2_colors'])

26

In [16]:
data.uns['mv_level2_colors'][24] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='mv_level2')

In [ ]:
sc.pl.umap(data, color='mv_level3')

In [19]:
len(data.uns['mv_level3_colors'])

62

In [20]:
data.uns['mv_level3_colors'][60] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='mv_level3')

## marker

In [5]:
sc.pp.normalize_total(data, target_sum=1e4)
sc.pp.log1p(data)

In [ ]:
sc.pl.umap(data, color='CD79A')

In [ ]:
sc.pl.umap(data, color=['CD3D', 'CD3E', 'CD8A'])

In [ ]:
sc.pl.umap(data, color=['CD68', 'CD163', 'CD14'])

In [ ]:
sc.pl.umap(data, color=['FGFBP2', 'CX3CR1'])

In [ ]:
sc.pl.umap(data, color=['CD34'])

# Benchmarker

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

2024-12-24 01:41:53.711605: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-24 01:41:53.711676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-24 01:41:53.713296: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-24 01:41:54.536855: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
adata = sc.read_h5ad('./processed/adata.h5ad')
adata

KeyboardInterrupt: 

## bm_1

In [7]:
embedding = np.load('processed/integrate/rna/rna_pca.npy')
embedding.shape

(1641866, 50)

In [8]:
embedding = np.load('processed/integrate/rna/rna_pca_harmony.npy')
embedding.shape

(1641866, 50)

In [9]:
embedding = np.load('processed/integrate/rna/rna_spectral2.npy')
embedding.shape

(1641866, 30)

In [10]:
embedding = np.load('processed/integrate/rna/rna_spectral2_harmony.npy')
embedding.shape

(1641866, 30)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

embedding = np.load('processed/integrate/rna/rna_pca.npy')
adata.obsm['rna_pca'] = embedding

embedding = np.load('processed/integrate/rna/rna_pca_harmony.npy')
adata.obsm['rna_pca_harmony'] = embedding

embedding = np.load('processed/integrate/rna/rna_spectral2.npy')
adata.obsm['rna_spectral2'] = embedding

embedding = np.load('processed/integrate/rna/rna_spectral2_harmony.npy')
adata.obsm['rna_spectral2_harmony'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['rna_pca','rna_pca_harmony', 
                         'rna_spectral2', 'rna_spectral2_harmony'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_1.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_1.csv", index=True)

with open("benchmarker_1.pkl", "wb") as f:
    pickle.dump(bm, f)

## bm_2

In [2]:
embedding = np.load('processed/integrate/atac/atac_lsi.npy')
embedding.shape

(1641866, 49)

In [3]:
embedding = np.load('processed/integrate/atac/atac_lsi_harmony.npy')
embedding.shape

(1641866, 49)

In [2]:
embedding = np.load('processed/integrate/atac/atac_spectral.npy')
embedding

array([[-5.61420972e-04, -8.33754999e-05,  1.04068810e-06, ...,
        -2.63750413e-06,  5.87927411e-07, -2.28004643e-06],
       [-6.14106134e-04, -9.48481247e-05,  2.22106251e-05, ...,
         1.31300654e-05, -1.15257317e-04,  4.06833017e-05],
       [-5.43535240e-04, -7.11941658e-05,  3.52090033e-05, ...,
         2.33622379e-05, -1.20049400e-05, -5.19739781e-06],
       ...,
       [-7.27061525e-04,  6.55474134e-04,  1.67679288e-04, ...,
        -8.45860546e-06, -2.84426751e-05, -5.09970778e-05],
       [-7.49635991e-04,  5.64711779e-04,  3.71539766e-04, ...,
        -4.28567997e-05, -4.10893776e-05, -4.04309971e-05],
       [-1.02735157e-03,  6.22516245e-04, -4.04112852e-05, ...,
        -5.46153299e-05,  4.12264533e-06,  3.60166646e-05]])

In [4]:
embedding = np.load('code/atac_spectral_harmony.npy')
embedding

array([[-6.03554483e-04,  6.28061026e-05,  4.16247805e-05, ...,
        -2.39081704e-06,  2.30872747e-05, -2.77471379e-06],
       [-6.56768767e-04,  4.72995468e-05,  6.55811344e-05, ...,
         1.33503425e-05, -9.27364649e-05,  4.01898777e-05],
       [-5.86033336e-04,  7.22079014e-05,  7.77130453e-05, ...,
         2.35907275e-05,  1.05092249e-05, -5.69120845e-06],
       ...,
       [-7.82090215e-04,  3.23790467e-04,  6.02057171e-07, ...,
         1.11001798e-05, -2.88825442e-05, -3.98207100e-05],
       [-8.11790181e-04,  2.32137615e-04,  1.62615509e-04, ...,
        -2.07337914e-05, -4.08731265e-05, -2.68734815e-05],
       [-1.08464268e-03,  2.90549836e-04, -2.20775389e-04, ...,
        -3.42423767e-05,  3.89110130e-06,  4.79490720e-05]])

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

embedding = np.load('processed/integrate/atac/atac_lsi.npy')
adata.obsm['atac_lsi'] = embedding

embedding = np.load('processed/integrate/atac/atac_lsi_harmony.npy')
adata.obsm['atac_lsi_harmony'] = embedding

embedding = np.load('processed/integrate/atac/atac_spectral.npy')
adata.obsm['atac_spectral'] = embedding

embedding = np.load('processed/integrate/atac/atac_spectral_harmony.npy')
adata.obsm['atac_spectral_harmony'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['atac_lsi','atac_lsi_harmony', 
                         'atac_spectral', 'atac_spectral_harmony'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_2.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_2.csv", index=True)

with open("benchmarker_2.pkl", "wb") as f:
    pickle.dump(bm, f)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

# embedding = np.load('processed/integrate/atac/atac_lsi.npy')
# adata.obsm['atac_lsi'] = embedding

# embedding = np.load('processed/integrate/atac/atac_lsi_harmony.npy')
# adata.obsm['atac_lsi_harmony'] = embedding

# embedding = np.load('processed/integrate/atac/atac_spectral.npy')
# adata.obsm['atac_spectral'] = embedding

embedding = np.load('processed/integrate/atac/atac_spectral_harmony.npy')
adata.obsm['atac_spectral_harmony'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['atac_spectral_harmony'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_2_4.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_2_4.csv", index=True)

with open("benchmarker_2_4.pkl", "wb") as f:
    pickle.dump(bm, f)

## bm_3

In [7]:
embedding = np.load('processed/integrate/multi_spectral/multi_spectral.npy')
embedding.shape

(1641866, 30)

In [8]:
embedding = np.load('processed/integrate/multi_spectral/multi_spectral_harmony.npy')
embedding.shape

(1641866, 30)

In [9]:
embedding = np.load('processed/integrate/multi_spectral/harmony_multi_spectral.npy')
embedding.shape

(1641866, 30)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

embedding = np.load('processed/integrate/multi_spectral/multi_spectral.npy')
adata.obsm['multi_spectral'] = embedding

embedding = np.load('processed/integrate/multi_spectral/multi_spectral_harmony.npy')
adata.obsm['multi_spectral_harmony'] = embedding

embedding = np.load('processed/integrate/multi_spectral/harmony_multi_spectral.npy')
adata.obsm['harmony_multi_spectral'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['multi_spectral','multi_spectral_harmony', 
                         'harmony_multi_spectral'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_3.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_3.csv", index=True)

with open("benchmarker_3.pkl", "wb") as f:
    pickle.dump(bm, f)

## bm_4

In [13]:
embedding = pd.read_csv('processed/integrate/wnn/wnn_m_umap.csv', index_col=0).to_numpy()
embedding.shape

(1641866, 2)

In [14]:
embedding = pd.read_csv('processed/integrate/wnn/wnn_h_m_umap.csv', index_col=0).to_numpy()
embedding.shape

(1641866, 2)

In [16]:
embedding = np.load('processed/integrate/multivi/mtv.npy')
embedding.shape

(1641866, 15)

In [24]:
embedding = pd.read_csv('processed/integrate/uinmf/uinmf.csv', index_col=0)
embedding = embedding.reindex(adata.obs_names.tolist()).to_numpy()
embedding.shape

(1641866, 30)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

embedding = pd.read_csv('processed/integrate/wnn/wnn_m_umap.csv', index_col=0).to_numpy()
adata.obsm['wnn_m_umap'] = embedding

embedding = pd.read_csv('processed/integrate/wnn/wnn_h_m_umap.csv', index_col=0).to_numpy()
adata.obsm['wnn_h_m_umap'] = embedding

embedding = np.load('processed/integrate/multivi/mtv.npy')
adata.obsm['mtv'] = embedding

embedding = pd.read_csv('processed/integrate/uinmf/uinmf.csv', index_col=0)
embedding = embedding.reindex(adata.obs_names.tolist()).to_numpy()
adata.obsm['uinmf'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['wnn_m_umap','wnn_h_m_umap', 
                         'mtv', 'uinmf'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_4.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_4.csv", index=True)

with open("benchmarker_4.pkl", "wb") as f:
    pickle.dump(bm, f)

## bm_5

In [25]:
import mudata as md

In [26]:
mdata = md.read("./processed/integrate/mofa/mofa.h5mu")
mdata

/home/qiaolibo/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/qiaolibo/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


MuData object with n_obs × n_vars = 328373 × 64000
  obs:	'batch'
  var:	'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
  uns:	'mofa', 'neighbors', 'umap'
  obsm:	'X_mofa', 'X_umap'
  varm:	'LFs'
  obsp:	'connectivities', 'distances'
  2 modalities
    rna:	328373 x 4000
      obs:	'dataset', 'barcode', 'sample'
      var:	'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'hvg', 'log1p'
      layers:	'counts'
    atac:	328373 x 60000
      obs:	'dataset', 'barcode', 'sample'
      var:	'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'hvg'
      layers:	'counts'

In [27]:
dat = adata[mdata.obs_names.to_list()]
dat

View of AnnData object with n_obs × n_vars = 328373 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType_raw', 'cellType_raw_level1', 'cellType_raw_level2', 'cellType_raw_level3', 'cellType_scarch_level3', 'cellType_actinn_level3', 'cellType_celltypist_level3', 'cellType_scmap_level3', 'cellType_singlecellnet_level3', 'cellType_scpred_level3', 'cellType_singler_level3', 'cellType_chetah_level3', 'cellType_scarch_level2', 'cellType_actinn_level2', 'cellType_celltypist_level2', 'cellType_scmap_level2', 'cellType_singlecellnet_level2', 'cellType_scpred_level2', 'cellType_singler_level2', 'cellType_chetah_level2', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'final_ann'

In [29]:
mdata.obsm['X_mofa'].shape

(328373, 10)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')
import mudata as md
mdata = md.read("./processed/integrate/mofa/mofa.h5mu")
dat = adata[mdata.obs_names.to_list()]
del adata
dat.obsm['X_mofa'] = mdata.obsm['X_mofa']
del mdata

bm = Benchmarker(
    dat,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['X_mofa'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_5.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_5.csv", index=True)

with open("benchmarker_5.pkl", "wb") as f:
    pickle.dump(bm, f)

## to_r

In [5]:
data = sc.read_h5ad('./processed/atac/data_new.h5ad')
data

AnnData object with n_obs × n_vars = 1641866 × 527459
    obs: 'dataset', 'barcode', 'sample'

In [6]:
data.X

<1641866x527459 sparse matrix of type '<class 'numpy.float32'>'
	with 9807077888 stored elements in Compressed Sparse Row format>

In [ ]:
sc.pp.subsample(data, 0.2)

In [14]:
data

AnnData object with n_obs × n_vars = 328373 × 527459
    obs: 'dataset', 'barcode', 'sample'

In [13]:
data.X

<328373x527459 sparse matrix of type '<class 'numpy.float32'>'
	with 1957539677 stored elements in Compressed Sparse Row format>

In [15]:
1957539677/2147483647

0.9115504463722699

In [16]:
from scipy.io import mmwrite

In [19]:
mmwrite("./processed/to_r/atac_matrix.mtx",data.X.T)

In [22]:
pd.DataFrame(data.obs_names).to_csv('./processed/to_r/cell.csv')

In [28]:
pd.DataFrame(data.var_names).to_csv('./processed/to_r/peak.csv')

In [3]:
adata = sc.read_h5ad('./processed/adata.h5ad')
adata

AnnData object with n_obs × n_vars = 1641866 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType_raw', 'cellType_raw_level1', 'cellType_raw_level2', 'cellType_raw_level3', 'cellType_scarch_level3', 'cellType_actinn_level3', 'cellType_celltypist_level3', 'cellType_scmap_level3', 'cellType_singlecellnet_level3', 'cellType_scpred_level3', 'cellType_singler_level3', 'cellType_chetah_level3', 'cellType_scarch_level2', 'cellType_actinn_level2', 'cellType_celltypist_level2', 'cellType_scmap_level2', 'cellType_singlecellnet_level2', 'cellType_scpred_level2', 'cellType_singler_level2', 'cellType_chetah_level2', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'final_ann'

In [23]:
adata = adata[data.obs_names.tolist()]
adata

View of AnnData object with n_obs × n_vars = 328373 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType_raw', 'cellType_raw_level1', 'cellType_raw_level2', 'cellType_raw_level3', 'cellType_scarch_level3', 'cellType_actinn_level3', 'cellType_celltypist_level3', 'cellType_scmap_level3', 'cellType_singlecellnet_level3', 'cellType_scpred_level3', 'cellType_singler_level3', 'cellType_chetah_level3', 'cellType_scarch_level2', 'cellType_actinn_level2', 'cellType_celltypist_level2', 'cellType_scmap_level2', 'cellType_singlecellnet_level2', 'cellType_scpred_level2', 'cellType_singler_level2', 'cellType_chetah_level2', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'final_ann'

In [24]:
adata.X

<328373x36601 sparse matrix of type '<class 'numpy.float32'>'
	with 593658843 stored elements in Compressed Sparse Row format>

In [25]:
593658843/2147483647

0.2764439411817323

In [30]:
mmwrite("./processed/to_r/rna_matrix.mtx",adata.X.T)

In [27]:
adata.obs.to_csv('./processed/to_r/meta.csv')

In [31]:
pd.DataFrame(adata.var_names).to_csv('./processed/to_r/gene.csv')

## Comparing single-omics-multi-omics

In [2]:
import pandas as pd

In [10]:
bm1 = pd.read_csv('./processed/bm/benchmark_1.csv', index_col=0)
bm1 = bm1.iloc[:bm1.shape[0]-1,:]
bm1

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,Silhouette batch,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
rna_pca,0.41855642,0.44144744126570873,0.2523853222680627,0.611333079636097,0.9999999724901639,0.8127227,0.014729378223419189,0.22030739303680544,0.5915391893618104,0,0.32785972871498315,0.5447444475310391,0.45799056000461674
rna_pca_harmony,0.35618883,0.4278118497153797,0.30221221978088375,0.6106389611959457,0.9984479408997756,0.7611409,0.038100242614746094,0.37545076617675327,0.5571646818215906,0.5401616268390249,0.45440364008420353,0.5390599610611033,0.5051974326703434
rna_spectral2,0.41240203,0.427282422945292,0.17888870501168389,0.595617026090622,0.9999999724901639,0.78889483,0.016442744731903075,0.2284681800391611,0.6003976337308213,0.5444792546902066,0.4357365290652678,0.5228380320687218,0.48799743086734015
rna_spectral2_harmony,0.36789563,0.3820857062861366,0.1834287787930647,0.543596088886261,0.9998106039487399,0.8023742,0.028581314086914063,0.3670960613274457,0.595879349369732,0.9667276296796633,0.5521317077190755,0.4953633621792912,0.518070700395205


In [14]:
bm2 = pd.read_csv('./processed/bm/benchmark_2.csv', index_col=0)
bm2 = bm2.iloc[:bm2.shape[0]-2,:]
bm2

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,Silhouette batch,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
atac_lsi,0.48963234,0.26471520372792706,0.02250865477946884,0.4795838613063097,0.9916020173292893,0.7454763,0.02976559638977051,0.2003031071737021,0.4323632078428429,0.5361920743279232,0.38882005824380206,0.4496084150856882,0.42529307234893376
atac_lsi_harmony,0.4867871,0.16755567019818457,0.036469608146593516,0.4777424428611994,0.9786141102130597,0.74256355,0.0373004150390625,0.35120848588844217,0.2561328783476425,0.9935839053460096,0.4761578460650089,0.42943378839722596,0.4481234114643391
atac_spectral,0.4068853,0.3511528396511818,0.10120459228766025,0.5416127406060696,0.9989867210388184,0.74737453,0.015846378803253174,0.13555346298606893,0.6089418281024244,0,0.301543240899736,0.4799684379380137,0.4085983591227026


In [13]:
bm24 = pd.read_csv('./processed/bm/benchmark_2_4.csv', index_col=0)
bm24 = bm24.iloc[:bm24.shape[0]-1,:]
bm24

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,Silhouette batch,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
atac_spectral_harmony,0.40078947,0.3155589515372473,0.1289119346102535,0.5564023070037365,0.998926960504972,0.79430145,0.01437654733657837,0.14820846058494977,0.5729776598900319,0.9733702260571415,0.5006468688242469,0.4801179246273447,0.4883295023061056


In [16]:
bm3 = pd.read_csv('./processed/bm/benchmark_3.csv', index_col=0)
bm3 = bm3.iloc[:bm3.shape[0]-1,:]
bm3

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,Silhouette batch,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
multi_spectral,0.41318965,0.4094629769764192,0.13876292333430443,0.596095860004425,0.9999999724901639,0.8002008,0.014734914302825928,0.2350502465527298,0.6306027208268224,0,0.33611774033031105,0.5115022764774444,0.441348462018591
multi_spectral_harmony,0.3753964,0.3769816546535219,0.15862530752648815,0.5744519233703613,0.9999999541502732,0.8100377,0.02304217338562012,0.31263474690889675,0.6040297295341777,0.9020331619493859,0.5303554968595529,0.49709104807814464,0.5103968275907079
harmony_multi_spectral,0.41999224,0.4117279782546773,0.18303735848593589,0.5741055607795715,0.9999999724901639,0.771402,0.013194782733917236,0.2067990828878777,0.6240770200950052,0,0.3230945774195441,0.5177726216587012,0.43990140396303834


In [19]:
bm4 = pd.read_csv('./processed/bm/benchmark_4.csv', index_col=0)
bm4 = bm4.iloc[:bm4.shape[0]-1,:]
bm4

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,Silhouette batch,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
wnn_m_umap,0.41721728,0.3864706507362712,0.12525509124649972,0.5744092091917992,0.9998954809629,0.5019624,0.025457546710968018,0.09390673705403223,0.49736659661094273,0,0.22373866074010557,0.5006495433156929,0.3898851902854579
wnn_h_m_umap,0.21579732,0.3742605004965026,0.10774991462195921,0.5423513613641262,0.9957902064690223,0.4123106,0.06432764053344726,0.3034406623915094,0.5177163182484906,0.00026065380956769344,0.2596111750527553,0.44718986059197763,0.37215838637628873
mtv,0.37128544,0.3916238773962591,0.1825420719699124,0.5574501007795334,0.9978420367607703,0.73703855,0.032737362384796145,0.2677338240611117,0.6597037980908469,0.4651471688505616,0.4324721412296789,0.5001487050888146,0.4730780795451603
uinmf,0.3525152,0.3485508114508969,0.2148940105998974,0.5493008084595203,0.9955439017369196,0.55939305,0.04564267635345459,0.2044600829488745,0.4451848833290941,0.622279974992854,0.37539213318214304,0.4921609446174003,0.44545342004329735


In [21]:
bm5 = pd.read_csv('./processed/bm/benchmark_5.csv', index_col=0)
bm5 = bm5.iloc[:bm5.shape[0]-1,:]
bm5

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,Silhouette batch,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
X_mofa,0.35293552,0.38735239382116915,0.19961987506327583,0.5621697157621384,0.9999999358103826,0.70584476,0.01755500555038452,0.20192927198570376,0.5264519207020231,0.9999996543963835,0.4903561225151192,0.5004154886503379,0.4963917421962504


In [22]:
bm = pd.concat([bm1, bm2, bm24, bm3, bm4, bm5])
bm

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,Silhouette batch,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
rna_pca,0.41855642,0.44144744126570873,0.2523853222680627,0.611333079636097,0.9999999724901639,0.8127227,0.014729378223419189,0.22030739303680544,0.5915391893618104,0,0.32785972871498315,0.5447444475310391,0.45799056000461674
rna_pca_harmony,0.35618883,0.4278118497153797,0.30221221978088375,0.6106389611959457,0.9984479408997756,0.7611409,0.038100242614746094,0.37545076617675327,0.5571646818215906,0.5401616268390249,0.45440364008420353,0.5390599610611033,0.5051974326703434
rna_spectral2,0.41240203,0.427282422945292,0.17888870501168389,0.595617026090622,0.9999999724901639,0.78889483,0.016442744731903075,0.2284681800391611,0.6003976337308213,0.5444792546902066,0.4357365290652678,0.5228380320687218,0.48799743086734015
rna_spectral2_harmony,0.36789563,0.3820857062861366,0.1834287787930647,0.543596088886261,0.9998106039487399,0.8023742,0.028581314086914063,0.3670960613274457,0.595879349369732,0.9667276296796633,0.5521317077190755,0.4953633621792912,0.518070700395205
atac_lsi,0.48963234,0.26471520372792706,0.02250865477946884,0.4795838613063097,0.9916020173292893,0.7454763,0.02976559638977051,0.2003031071737021,0.4323632078428429,0.5361920743279232,0.38882005824380206,0.4496084150856882,0.42529307234893376
atac_lsi_harmony,0.4867871,0.16755567019818457,0.036469608146593516,0.4777424428611994,0.9786141102130597,0.74256355,0.0373004150390625,0.35120848588844217,0.2561328783476425,0.9935839053460096,0.4761578460650089,0.42943378839722596,0.4481234114643391
atac_spectral,0.4068853,0.3511528396511818,0.10120459228766025,0.5416127406060696,0.9989867210388184,0.74737453,0.015846378803253174,0.13555346298606893,0.6089418281024244,0,0.301543240899736,0.4799684379380137,0.4085983591227026
atac_spectral_harmony,0.40078947,0.3155589515372473,0.1289119346102535,0.5564023070037365,0.998926960504972,0.79430145,0.01437654733657837,0.14820846058494977,0.5729776598900319,0.9733702260571415,0.5006468688242469,0.4801179246273447,0.4883295023061056
multi_spectral,0.41318965,0.4094629769764192,0.13876292333430443,0.596095860004425,0.9999999724901639,0.8002008,0.014734914302825928,0.2350502465527298,0.6306027208268224,0,0.33611774033031105,0.5115022764774444,0.441348462018591


In [32]:
bm.index = ['rna_pca', 'rna_pca_harmony', 'rna_spectral2', 'rna_spectral2_harmony',
       'atac_lsi', 'atac_lsi_harmony', 'atac_spectral',
       'atac_spectral_harmony', 'multi_spectral', 'multi_spectral_harmony',
       'harmony_multi_spectral', 'wnn_m_umap', 'wnn_h_m_umap', 'mtv', 'uinmf',
       'X_mofa']

In [33]:
bm_bio = bm.loc[['rna_pca','atac_lsi','wnn_m_umap'],['Batch correction','Bio conservation','Total']]
bm_bio

,Batch correction,Bio conservation,Total
rna_pca,0.32785972871498315,0.5447444475310391,0.45799056000461674
atac_lsi,0.38882005824380206,0.4496084150856882,0.42529307234893376
wnn_m_umap,0.22373866074010557,0.5006495433156929,0.3898851902854579


In [34]:
bm_bio = bm.loc[['rna_pca_harmony','atac_lsi_harmony','wnn_h_m_umap'],['Batch correction','Bio conservation','Total']]
bm_bio

,Batch correction,Bio conservation,Total
rna_pca_harmony,0.45440364008420353,0.5390599610611033,0.5051974326703434
atac_lsi_harmony,0.4761578460650089,0.42943378839722596,0.4481234114643391
wnn_h_m_umap,0.2596111750527553,0.44718986059197763,0.37215838637628873


In [35]:
bm_bio = bm.loc[['rna_spectral2','atac_spectral','multi_spectral'],['Batch correction','Bio conservation','Total']]
bm_bio

,Batch correction,Bio conservation,Total
rna_spectral2,0.4357365290652678,0.5228380320687218,0.48799743086734015
atac_spectral,0.301543240899736,0.4799684379380137,0.4085983591227026
multi_spectral,0.33611774033031105,0.5115022764774444,0.441348462018591


In [36]:
bm_bio = bm.loc[['rna_spectral2_harmony','atac_spectral_harmony','multi_spectral_harmony','harmony_multi_spectral'],['Batch correction','Bio conservation','Total']]
bm_bio

,Batch correction,Bio conservation,Total
rna_spectral2_harmony,0.5521317077190755,0.4953633621792912,0.518070700395205
atac_spectral_harmony,0.5006468688242469,0.4801179246273447,0.4883295023061056
multi_spectral_harmony,0.5303554968595529,0.49709104807814464,0.5103968275907079
harmony_multi_spectral,0.3230945774195441,0.5177726216587012,0.43990140396303834
